# 🥇 Gold Futures Price Analysis & Prediction
### Candlestick analytics with `mplfinance`, Regression & Random Forest forecasting, and a Power BI-ready data layer

**Dataset:** `finalgolddata.csv` — daily gold OHLCV, 30 Aug 2000 → 3 Mar 2026  
**Tools:** Python · pandas · NumPy · matplotlib · mplfinance · scikit-learn · SciPy

**Contents**
1. Introduction  
2. Problem Questions  
3. Objectives  
4. Data Collection  
5. Data Cleaning  
6. Data Visualization (mplfinance + advanced analytics)  
7. Methods and Result Analysis (Prediction Modeling, `r2_score` evaluation)  
8. Exporting the Analytics Layer for the Power BI Dashboard  
9. Discussion and Conclusion  
10. References

---
## 1. Introduction

Gold has been used as a store of value for thousands of years, and in modern financial markets it plays three roles at once: a **safe-haven asset** that investors buy during crises, an **inflation and currency hedge**, and a heavily traded **speculative instrument** through futures and ETFs. Its price reacts to real interest rates, the US dollar, central-bank buying, geopolitical shocks and investor sentiment. Between 2000 and 2026 gold passed through several distinct regimes: the early-2000s bull market, the 2008 financial crisis, the 2011 peak, a long consolidation, the COVID-19 shock of 2020, and a sharp acceleration in the mid-2020s. Understanding how its price behaves across these regimes matters for investors, risk managers and policy makers alike.

This project analyses roughly 25 years of **daily gold futures data** (Open, High, Low, Close, Volume). We first audit and clean the data, then explore it visually with `mplfinance` (candlestick, OHLC, hollow-candle, monthly-candlestick and Renko charts) and with statistical analytics (calendar-year returns, drawdowns, tail risk, volatility clustering, a monthly-returns heat-map and liquidity). Finally, we build **two families of predictive models — a regression model and a Random Forest regressor (plus Gradient Boosting as an extension)** — and evaluate them with `r2_score` under a strict chronological train/test split. The results are then exported as tidy tables so that the same analysis can be reproduced as an interactive **Power BI dashboard**.

---
## 2. Problem Questions

1. **Data quality:** Is the raw gold price file reliable enough for analysis? Are there missing values, duplicates, impossible OHLC combinations or suspicious outliers?
2. **Long-term behaviour:** How has the gold price evolved from 2000 to 2026, and which years delivered the strongest gains and the deepest drawdowns?
3. **Risk profile:** Are gold returns normally distributed? How large are the tail risks (VaR / CVaR), and does volatility cluster in time?
4. **Technical structure:** What do candlestick, OHLC, hollow-candle, monthly and Renko views and moving averages reveal about trend and momentum?
5. **Seasonality:** Does the monthly-returns heat-map (and an ANOVA test) show a repeatable month-of-year effect?
6. **Predictability:** Can a regression model and a Random Forest predict the next-day gold price or return better than a naïve "tomorrow = today" benchmark?
7. **Model validity:** Why can a model report an R² above 0.99 on prices and still be useless? What is the right way to evaluate it?

---
## 3. Objectives

| # | Objective | Success criterion |
|---|-----------|-------------------|
| O1 | Collect and describe the gold OHLCV dataset | Schema, date range, row count and summary statistics documented |
| O2 | Clean and validate the data | Zero missing values, duplicates or inconsistent OHLC bars after cleaning; every repair logged |
| O3 | Visualise price action with several `mplfinance` chart types | At least 5 distinct chart types (candlestick, OHLC, hollow, monthly candlestick, Renko) |
| O4 | Perform advanced analytics | Annual returns, drawdowns, tail-risk metrics, volatility clustering and monthly heat-map computed |
| O5 | Build ≥ 2 prediction models | Linear Regression **and** Random Forest Regression (plus Ridge / Gradient Boosting as extensions) |
| O6 | Evaluate rigorously | `r2_score`, RMSE, MAE and directional accuracy on a chronological hold-out set, compared with a naïve baseline and time-series cross-validation |
| O7 | Prepare a Power BI analytics layer | Tidy CSV tables exported for the dashboard |
| O8 | Draw honest conclusions | Limitations of price prediction clearly stated |

### 0. Environment setup
Run this cell first. It installs `mplfinance` only if it is missing (e.g. on a fresh Colab runtime).

In [ ]:
import importlib.util, subprocess, sys

if importlib.util.find_spec("mplfinance") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mplfinance"])

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import mplfinance as mpf
from scipy import stats

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, cross_val_score
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold", "axes.titlesize": 12})

RANDOM_STATE = 42
OUT_DIR = Path("powerbi_data")
OUT_DIR.mkdir(exist_ok=True)
GOLD = "#C9A227"      # accent colour used across charts
print("pandas", pd.__version__, "| mplfinance", mpf.__version__)

---
## 4. Data Collection

The dataset is a daily time series of **gold futures prices in US dollars per troy ounce**. The date range, price level and column layout are consistent with the COMEX gold futures contract (continuous series), but the file itself carries no metadata, so please confirm the exact provider when you cite it.

| Column | Meaning |
|--------|---------|
| `Date` | Trading day, stored as `dd-mm-yy` text |
| `Open` | First traded price of the day |
| `High` | Highest traded price of the day |
| `Low` | Lowest traded price of the day |
| `Close` | Last traded price of the day |
| `Volume` | Number of contracts traded |

The cell below looks for the file in the working directory, common Colab paths, and finally falls back to an upload dialog.

In [ ]:
CANDIDATES = [
    Path("finalgolddata.csv"),
    Path("/mnt/user-data/uploads/finalgolddata.csv"),
    Path("/content/finalgolddata.csv"),
    Path("/content/drive/MyDrive/finalgolddata.csv"),
]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)

if DATA_PATH is None:                       # Colab upload fallback
    try:
        from google.colab import files
        uploaded = files.upload()
        DATA_PATH = Path(next(iter(uploaded)))
    except ImportError:
        raise FileNotFoundError("Place finalgolddata.csv next to this notebook and re-run.")

raw = pd.read_csv(DATA_PATH)
print(f"Loaded {DATA_PATH}  ->  {raw.shape[0]:,} rows x {raw.shape[1]} columns\n")
display(raw.head(8))
display(raw.tail(5))

In [ ]:
print("Data types:\n", raw.dtypes, "\n")
display(raw.describe().T)

---
## 5. Data Cleaning

Cleaning steps (each one is logged in a `cleaning_log` so the process is auditable):

1. Parse `Date` from `dd-mm-yy`, sort chronologically, set it as the index  
2. Check missing values and duplicate dates  
3. Check non-positive prices  
4. **Validate OHLC logic** (`High` ≥ max(Open, Close, Low); `Low` ≤ min(Open, Close, High)) and repair violations  
5. Flag zero-volume and zero-range ("flat") bars rather than deleting them  
6. Inspect gaps between trading days  
7. Screen return outliers (kept if they correspond to real market events)

In [ ]:
cleaning_log = []
def log(step, finding, action):
    cleaning_log.append({"Step": step, "Finding": finding, "Action": action})

df = raw.copy()
df.columns = df.columns.str.strip().str.title()

# 1. Dates
df["Date"] = pd.to_datetime(df["Date"], format="%d-%m-%y", errors="coerce")
bad_dates = int(df["Date"].isna().sum())
df = df.dropna(subset=["Date"]).sort_values("Date").set_index("Date")
log("Date parsing", f"{bad_dates} unparseable dates", "Parsed dd-mm-yy, sorted ascending, set as index")

# 2. Missing & duplicates
n_missing = int(df.isna().sum().sum())
n_dup = int(df.index.duplicated().sum())
df = df[~df.index.duplicated(keep="last")].dropna()
log("Missing / duplicates", f"{n_missing} missing cells, {n_dup} duplicate dates", "None needed" if n_missing + n_dup == 0 else "Removed")

# 3. Non-positive prices
price_cols = ["Open", "High", "Low", "Close"]
n_nonpos = int((df[price_cols] <= 0).any(axis=1).sum())
df = df[(df[price_cols] > 0).all(axis=1)]
log("Non-positive prices", f"{n_nonpos} rows", "None needed" if n_nonpos == 0 else "Removed")

# 4. OHLC logical consistency
hi_ok = df["High"] >= df[["Open", "Low", "Close"]].max(axis=1)
lo_ok = df["Low"] <= df[["Open", "High", "Close"]].min(axis=1)
n_incons = int((~(hi_ok & lo_ok)).sum())
first_bad = df.index[~(hi_ok & lo_ok)]
df["High"] = df[price_cols].max(axis=1)
df["Low"] = df[price_cols].min(axis=1)
log("OHLC consistency", f"{n_incons} bars violate High/Low bounds (first: {first_bad.min().date()}, last: {first_bad.max().date()})",
    "Set High = max(O,H,L,C) and Low = min(O,H,L,C)")

# 5. Flags (kept, not deleted)
df["Zero_Volume"] = (df["Volume"] == 0)
df["Flat_Bar"] = (df["High"] == df["Low"])
log("Zero volume", f"{int(df['Zero_Volume'].sum())} days ({df['Zero_Volume'].mean():.1%})", "Flagged in column Zero_Volume; kept")
log("Flat bars (H = L)", f"{int(df['Flat_Bar'].sum())} days, mostly in the illiquid early years", "Flagged in column Flat_Bar; kept")

# 6. Gaps between observations
gaps = df.index.to_series().diff().dt.days.dropna()
log("Date gaps", f"max gap {int(gaps.max())} days; {int((gaps > 4).sum())} gaps > 4 days", "Weekends/holidays only - no action")

# 7. Return outliers
df["Return"] = df["Close"].pct_change()
z = (df["Return"] - df["Return"].mean()) / df["Return"].std()
outliers = df.loc[z.abs() > 4, ["Close", "Return"]]
log("Return outliers (|z| > 4)", f"{len(outliers)} days", "Kept - correspond to genuine market events (see table below)")

gold = df.drop(columns="Return").copy()
display(pd.DataFrame(cleaning_log))
print("\nLargest daily moves flagged as outliers:")
display(outliers.assign(Return_pct=outliers["Return"] * 100).sort_values("Return_pct").drop(columns="Return"))

In [ ]:
# Post-cleaning validation
assert gold.index.is_monotonic_increasing and gold.index.is_unique
assert gold[price_cols].notna().all().all()
assert (gold["High"] >= gold[["Open", "Low", "Close"]].max(axis=1)).all()
assert (gold["Low"] <= gold[["Open", "High", "Close"]].min(axis=1)).all()

print(f"Clean dataset: {len(gold):,} rows | {gold.index.min().date()} -> {gold.index.max().date()}")
display(gold.describe().T)

---
## 6. Data Visualization

### 6.1 Technical indicators
We first compute the indicators used in the charts and later as model features: simple moving averages, Bollinger Bands (20, 2σ), Wilder's RSI (14) and MACD (12, 26, 9).

In [ ]:
def add_indicators(d):
    d = d.copy()
    c = d["Close"]
    for w in (5, 10, 20, 50, 200):
        d[f"SMA_{w}"] = c.rolling(w).mean()
    # Bollinger Bands
    sd20 = c.rolling(20).std()
    d["BB_Mid"], d["BB_Up"], d["BB_Low"] = d["SMA_20"], d["SMA_20"] + 2 * sd20, d["SMA_20"] - 2 * sd20
    d["BB_PctB"] = (c - d["BB_Low"]) / (d["BB_Up"] - d["BB_Low"])
    # RSI (Wilder smoothing)
    delta = c.diff()
    gain = delta.clip(lower=0).ewm(alpha=1 / 14, adjust=False).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1 / 14, adjust=False).mean()
    d["RSI_14"] = 100 - 100 / (1 + gain / loss)
    # MACD
    ema12, ema26 = c.ewm(span=12, adjust=False).mean(), c.ewm(span=26, adjust=False).mean()
    d["MACD"] = ema12 - ema26
    d["MACD_Signal"] = d["MACD"].ewm(span=9, adjust=False).mean()
    d["MACD_Hist"] = d["MACD"] - d["MACD_Signal"]
    # Returns, volatility, drawdown
    d["Return"] = c.pct_change()
    d["LogRet"] = np.log(c).diff()
    d["Vol_21d_Ann"] = d["LogRet"].rolling(21).std() * np.sqrt(252)
    d["Drawdown"] = c / c.cummax() - 1
    return d

gold = add_indicators(gold)
display(gold.tail(5).T)

### 6.2 Candlestick chart with moving averages and volume (last 120 trading days)
Each candle shows Open/High/Low/Close; green = close above open, red = close below open. Wicks show the intraday range.

In [ ]:
mc = mpf.make_marketcolors(up="#26a69a", down="#ef5350", inherit=True)
STYLE = mpf.make_mpf_style(base_mpf_style="yahoo", marketcolors=mc, gridstyle=":", gridcolor="#d0d0d0")
COMMON = dict(style=STYLE, warn_too_much_data=100000, figratio=(20 , 10), figscale=1.0)

recent = gold.tail(120)
mpf.plot(recent, type="candle", mav=(10, 20), volume=True,
         title="Gold futures - candlestick, 10/20-day MAs and volume (last 120 days)",
         ylabel="Price (USD/oz)", ylabel_lower="Contracts", **COMMON)

### 6.3 OHLC bar chart (last 60 days)

In [ ]:
mpf.plot(gold.tail(60), type="ohlc", volume=True, mav=(5, 10),
         title="Gold futures - OHLC bars (last 60 days)", ylabel="Price (USD/oz)", **COMMON)

### 6.4 Hollow-and-filled candles (last 120 days)
Hollow candles = close above open; filled = close below open. Colour = close vs *previous* close.

In [ ]:
mpf.plot(gold.tail(120), type="hollow_and_filled", mav=(20,),
         title="Gold futures - hollow & filled candles (last 120 days)", ylabel="Price (USD/oz)", **COMMON)

### 6.5 Monthly candlesticks (full history)
Resampling daily bars into **monthly OHLCV** makes multi-year cycles easy to read.

In [ ]:
monthly = gold.resample("MS").agg({"Open": "first", "High": "max", "Low": "min", "Close": "last", "Volume": "sum"}).dropna()
mpf.plot(monthly, type="candle", mav=(12,), volume=True,
         title="Gold futures - monthly candlesticks with 12-month MA", ylabel="Price (USD/oz)", **COMMON)

### 6.6 Renko chart (last ~3 years)
Renko bricks ignore time and only move when price moves by a fixed amount (here **USD 50**), filtering out noise.

In [ ]:
mpf.plot(gold.tail(750), type="renko", renko_params=dict(brick_size=50),
         title="Gold futures - Renko chart (brick = USD 50)", ylabel="Price (USD/oz)",
         style=STYLE, warn_too_much_data=100000, figratio=(16, 8))

### 6.7 Advanced analytics
#### (a) Annual performance and drawdown

In [ ]:
yearly = gold.groupby(gold.index.year).agg(Open=("Open", "first"), High=("High", "max"), Low=("Low", "min"),
                                           Close=("Close", "last"), Avg_Volume=("Volume", "mean"))
prev_close = yearly["Close"].shift(1).fillna(yearly["Open"])
yearly["Return_pct"] = (yearly["Close"] / prev_close - 1) * 100
yearly["Ann_Vol_pct"] = gold.groupby(gold.index.year)["LogRet"].std() * np.sqrt(252) * 100
yearly["Max_DD_pct"] = gold.groupby(gold.index.year)["Drawdown"].min() * 100
yearly.index.name = "Year"

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
colors = ["#26a69a" if v >= 0 else "#ef5350" for v in yearly["Return_pct"]]
axes[0].bar(yearly.index, yearly["Return_pct"], color=colors)
axes[0].set(title="Calendar-year return (%)", xlabel="Year")
axes[1].fill_between(gold.index, gold["Drawdown"] * 100, 0, color="#ef5350", alpha=0.6)
axes[1].set(title="Drawdown from running peak (%)", xlabel="Date")
plt.tight_layout(); plt.show()
display(yearly.round(2))

#### (b) Return statistics, tail risk and normality
This cell computes the log-returns `r`, CAGR, VaR/CVaR, skewness/kurtosis, the Jarque-Bera normality test and a month-of-year ANOVA. Later sections reuse its results.

In [ ]:
r = gold["LogRet"].dropna()
mu, sigma = r.mean(), r.std()
jb_stat, jb_p = stats.jarque_bera(r)


var95, var99 = np.percentile(r, 5), np.percentile(r, 1)
cvar95 = r[r <= var95].mean()
n_years = (gold.index[-1] - gold.index[0]).days / 365.25
cagr = (gold["Close"].iloc[-1] / gold["Close"].iloc[0]) ** (1 / n_years) - 1
risk_table = pd.Series({
    "CAGR": f"{cagr:.2%}",
    "Annualised volatility": f"{sigma * np.sqrt(252):.2%}",
    "Sharpe ratio (rf = 0)": f"{mu * 252 / (sigma * np.sqrt(252)):.2f}",
    "Max drawdown": f"{gold['Drawdown'].min():.2%}",
    "Skewness": f"{stats.skew(r):.3f}",
    "Excess kurtosis": f"{stats.kurtosis(r):.2f}",
    "Jarque-Bera p-value": f"{jb_p:.2e}",
    "VaR 95% (1-day)": f"{var95:.2%}",
    "VaR 99% (1-day)": f"{var99:.2%}",
    "CVaR 95% (1-day)": f"{cvar95:.2%}",
    "Positive days": f"{(r > 0).mean():.1%}",
    "Best / worst day": f"{r.max():.2%} / {r.min():.2%}",
}, name="Value").to_frame()
display(risk_table)

# month-of-year effect test (one-way ANOVA on daily % returns)
s = gold["Return"].dropna() * 100
f_stat, p_anova = stats.f_oneway(*[g.values for _, g in s.groupby(s.index.month)])
print(f"\nANOVA across calendar months: F = {f_stat:.2f}, p = {p_anova:.3f}  ->  "
      f"{'some evidence of month effects' if p_anova < 0.05 else 'no statistically significant month effect'}")

#### (c) Volatility clustering
If returns themselves are almost uncorrelated but **absolute returns** are strongly autocorrelated, volatility is predictable even when direction is not.

In [ ]:
lags = range(1, 31)
acf_ret = [r.autocorr(l) for l in lags]
acf_abs = [r.abs().autocorr(l) for l in lags]
conf = 1.96 / np.sqrt(len(r))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
axes[0].bar(lags, acf_ret, color="steelblue"); axes[0].set_title("Autocorrelation of returns")
axes[1].bar(lags, acf_abs, color="darkorange"); axes[1].set_title("Autocorrelation of |returns| (volatility)")
for ax in axes:
    ax.axhline(conf, ls="--", c="r", lw=0.8); ax.axhline(-conf, ls="--", c="r", lw=0.8); ax.set_xlabel("Lag (days)")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(13, 3.8))
ax.plot(gold.index, gold["Vol_21d_Ann"] * 100, color="purple", lw=0.9)
ax.axhline(gold["Vol_21d_Ann"].median() * 100, color="k", ls="--", lw=0.8, label="Median")
ax.set(title="Rolling 21-day annualised volatility (%)", ylabel="%"); ax.legend()
plt.show()
print(f"Mean |autocorr| of returns (lags 1-30): {np.mean(np.abs(acf_ret)):.3f}  vs  of |returns|: {np.mean(acf_abs):.3f}")

#### (d) Monthly returns heat-map (year x month)

In [ ]:
m_ret = gold["Close"].resample("MS").last().pct_change().dropna() * 100
heat = m_ret.to_frame("ret").assign(Year=m_ret.index.year, Month=m_ret.index.month).pivot(index="Year", columns="Month", values="ret")
heat.columns = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"][:heat.shape[1]]
plt.figure(figsize=(13, 8))
sns.heatmap(heat, annot=True, fmt=".1f", cmap="RdYlGn", center=0, linewidths=0.4, cbar_kws={"label": "Monthly return (%)"})
plt.title("Gold - monthly returns (%)"); plt.show()

#### (e) Volume, liquidity and price range
Early years have very little volume (the series is thin before ~2003), so volume must be used with care as a model feature.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
vol_y = gold.groupby(gold.index.year)["Volume"].mean()
axes[0].bar(vol_y.index, vol_y.values, color="steelblue"); axes[0].set(title="Average daily volume by year", yscale="log")
rng = ((gold["High"] - gold["Low"]) / gold["Close"] * 100).rolling(21).mean()
axes[1].plot(gold.index, rng, color="teal", lw=0.9); axes[1].set(title="21-day average daily range (% of close)")
plt.tight_layout(); plt.show()

---
## 7. Methods and Result Analysis

### 7.1 Modelling strategy

Predicting a trending price series is deceptively easy to get *wrong*, so the modelling is done in two stages:

| Stage | Target | Why |
|-------|--------|-----|
| **A. Price-level models** | Next-day **Close** | The "obvious" formulation. It will be shown to give very high R² for Linear Regression but to **break down for Random Forest**, because tree models cannot predict values outside the price range seen in training (gold rose from ~$270 to over $5,000). |
| **B. Return-level models** | Next-day **log-return**, converted back to a price | The statistically correct formulation: returns are roughly stationary, so Random Forest can be used fairly. |

**Safeguards against data leakage:** all features at day *t* use only information available at the close of day *t*; the target is day *t + 1*; the data are split **chronologically** (first 80 % train, last 20 % test) — never shuffled; hyper-parameters are tuned with `TimeSeriesSplit` on the training set only.

**Algorithms:** (i) Linear Regression (+ Ridge), (ii) Random Forest Regressor, (iii) Gradient Boosting Regressor as an advanced extension.  
**Metric:** `r2_score` (primary), with RMSE, MAE, MAPE and directional accuracy as supporting measures, each compared with a **naïve baseline** (tomorrow's price = today's price).

### 7.2 Feature engineering

In [ ]:
def make_features(d):
    c, lc = d["Close"], np.log(d["Close"])
    f = pd.DataFrame(index=d.index)
    f["ret_1"] = d["LogRet"]
    for k in (2, 3, 5, 10, 20):
        f[f"ret_{k}"] = lc.diff(k)                                   # momentum over k days
    for w in (5, 10, 20, 50):
        f[f"sma_gap_{w}"] = c / d[f"SMA_{w}"] - 1                    # distance from moving average
    f["vol_10"] = d["LogRet"].rolling(10).std()
    f["vol_20"] = d["LogRet"].rolling(20).std()
    f["rsi_14"] = d["RSI_14"]
    f["macd_hist_n"] = d["MACD_Hist"] / c                            # scale-free MACD histogram
    f["bb_pctb"] = d["BB_PctB"]
    f["gap"] = d["Open"] / c.shift(1) - 1                            # overnight gap
    f["hl_range"] = (d["High"] - d["Low"]) / c
    f["oc_body"] = (d["Close"] - d["Open"]) / d["Open"]
    f["close_pos"] = (d["Close"] - d["Low"]) / (d["High"] - d["Low"]).replace(0, np.nan)
    f["log_volume"] = np.log1p(d["Volume"])
    f["dow"] = d.index.dayofweek
    f["month_sin"] = np.sin(2 * np.pi * d.index.month / 12)
    f["month_cos"] = np.cos(2 * np.pi * d.index.month / 12)
    f["close"] = c
    f["target_ret"] = d["LogRet"].shift(-1)                          # next-day log-return
    f["target_close"] = c.shift(-1)                                  # next-day close
    return f

feat = make_features(gold)
latest_row = feat.iloc[[-1]]                       # newest day (no target yet) - used for a live forecast later
feat = feat.dropna(subset=[c for c in feat.columns if c not in ("target_ret", "target_close", "close_pos")])
feat["close_pos"] = feat["close_pos"].fillna(0.5)
feat = feat.dropna()

FEATURES = [c for c in feat.columns if c not in ("close", "target_ret", "target_close")]
split = int(len(feat) * 0.8)
train, test = feat.iloc[:split], feat.iloc[split:]
print(f"Samples: {len(feat):,} | features: {len(FEATURES)}")
print(f"Train: {train.index.min().date()} -> {train.index.max().date()}  ({len(train):,} rows, max close {train['close'].max():,.0f})")
print(f"Test : {test.index.min().date()} -> {test.index.max().date()}  ({len(test):,} rows, max close {test['close'].max():,.0f})")
print(f"\nTest-set prices go up to {test['close'].max() / train['close'].max():.1f}x the highest price ever seen in training -> extrapolation problem for tree models.")

### 7.3 Stage A — Price-level models (Regression vs Random Forest)
Inputs: today's Open, High, Low, Close and 5/20/50-day moving averages (all in USD). Target: tomorrow's Close.

In [ ]:
LEVEL_FEATS = ["Open", "High", "Low", "Close", "SMA_5", "SMA_20", "SMA_50"]
lvl = gold[LEVEL_FEATS].join(gold["Close"].shift(-1).rename("target")).dropna()
lvl_tr, lvl_te = lvl.loc[:train.index.max()], lvl.loc[test.index.min():]

level_models = {
    "Linear Regression (price)": LinearRegression(),
    "Random Forest (price)": RandomForestRegressor(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE),
}
level_rows, level_preds = [], {}
for name, m in level_models.items():
    m.fit(lvl_tr[LEVEL_FEATS], lvl_tr["target"])
    p = m.predict(lvl_te[LEVEL_FEATS]); level_preds[name] = p
    level_rows.append({"Model": name,
                       "R2_train": r2_score(lvl_tr["target"], m.predict(lvl_tr[LEVEL_FEATS])),
                       "R2_test": r2_score(lvl_te["target"], p),
                       "RMSE_test": mean_squared_error(lvl_te["target"], p) ** 0.5,
                       "MAE_test": mean_absolute_error(lvl_te["target"], p)})
naive_p = lvl_te["Close"].values
level_rows.append({"Model": "Naive (tomorrow = today)", "R2_train": np.nan, "R2_test": r2_score(lvl_te["target"], naive_p),
                   "RMSE_test": mean_squared_error(lvl_te["target"], naive_p) ** 0.5,
                   "MAE_test": mean_absolute_error(lvl_te["target"], naive_p)})
level_results = pd.DataFrame(level_rows).set_index("Model")
display(level_results)

fig, ax = plt.subplots(figsize=(13, 4.8))
ax.plot(lvl_te.index, lvl_te["target"], color="black", lw=1.4, label="Actual")
ax.plot(lvl_te.index, level_preds["Linear Regression (price)"], color="#1f77b4", lw=1, label="Linear Regression")
ax.plot(lvl_te.index, level_preds["Random Forest (price)"], color="#d62728", lw=1, label="Random Forest")
ax.axhline(lvl_tr["target"].max(), color="grey", ls="--", label="Highest price seen in training")
ax.set(title="Stage A - next-day close: Random Forest flat-lines above the training range", ylabel="USD / oz"); ax.legend()
plt.show()

**Reading Stage A.** Linear Regression keeps a very high R² because it can extrapolate a straight-line relationship (tomorrow's close ≈ today's close). The Random Forest scores well on training data but collapses on the test set: every tree predicts an *average of training prices*, so it cannot follow gold above the highest level it has ever seen. This is a structural weakness of tree ensembles on trending, non-stationary levels — **not** a tuning problem — and it motivates Stage B.

### 7.4 Stage B — Return-level models
Target: next-day **log-return**. Predicted returns are converted back to prices with  
`predicted_close(t+1) = close(t) × exp(predicted_return)`.  
Models: Linear Regression, Ridge (L2-regularised), Random Forest, Gradient Boosting.

In [ ]:
X_tr, y_tr = train[FEATURES], train["target_ret"]
X_te, y_te = test[FEATURES], test["target_ret"]

models = {
    "Linear Regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge Regression":  make_pipeline(StandardScaler(), Ridge(alpha=100)),
    "Random Forest":     RandomForestRegressor(n_estimators=400, min_samples_leaf=20, max_features=0.5,
                                               n_jobs=-1, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, max_depth=2, learning_rate=0.03,
                                                   subsample=0.7, random_state=RANDOM_STATE),
}

def score_row(name, y_true_ret, y_pred_ret, close_t):
    p_true = close_t * np.exp(y_true_ret); p_pred = close_t * np.exp(y_pred_ret)
    return {"Model": name,
            "R2_return": r2_score(y_true_ret, y_pred_ret),
            "R2_price": r2_score(p_true, p_pred),
            "RMSE_price": mean_squared_error(p_true, p_pred) ** 0.5,
            "MAE_price": mean_absolute_error(p_true, p_pred),
            "MAPE_%": np.mean(np.abs((p_true - p_pred) / p_true)) * 100,
            "Directional_Acc_%": (np.nan if np.all(y_pred_ret == 0) else np.mean(np.sign(y_pred_ret) == np.sign(y_true_ret)) * 100)}

rows, pred_ret = [], {}
for name, m in models.items():
    m.fit(X_tr, y_tr)
    pred_ret[name] = m.predict(X_te)
    rows.append(score_row(name, y_te.values, pred_ret[name], test["close"].values))

# Baselines
pred_ret["Naive (zero return)"] = np.zeros(len(test))
rows.append(score_row("Naive (zero return)", y_te.values, pred_ret["Naive (zero return)"], test["close"].values))
pred_ret["Always-long (train mean)"] = np.full(len(test), y_tr.mean())
rows.append(score_row("Always-long (train mean)", y_te.values, pred_ret["Always-long (train mean)"], test["close"].values))

results = pd.DataFrame(rows).set_index("Model").sort_values("R2_return", ascending=False)
display(results)

**How to read this table.**  
* `R2_return` is the honest score: how much of tomorrow's *return* variance the model explains. A value around 0 means "no better than guessing the average".  
* `R2_price` is almost identical for **every** model, including the naïve baseline, because yesterday's price already explains >99.8 % of tomorrow's price. A high price-level R² therefore says nothing about forecasting skill.  
* `Directional_Acc_%` is the share of days on which the sign of the move is right. Compare it with the **"Always-long" baseline**, which predicts "up" every day: gold rose on ~55 % of test days, so any model must beat ~55 %, not 50 %, to show real directional skill. (The naïve zero-return model predicts no direction, so it is shown as blank.)

### 7.5 Time-series cross-validation and hyper-parameter tuning (training data only)

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

cv_rows = []
for name in ["Linear Regression", "Ridge Regression", "Random Forest", "Gradient Boosting"]:
    sc = cross_val_score(models[name], X_tr, y_tr, cv=tscv, scoring="r2", n_jobs=1)
    cv_rows.append({"Model": name, "CV_R2_mean": sc.mean(), "CV_R2_std": sc.std(), "Folds": np.round(sc, 4).tolist()})
display(pd.DataFrame(cv_rows).set_index("Model"))

grid = GridSearchCV(
    RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
    param_grid={"max_depth": [3, 5, 8], "min_samples_leaf": [20, 50], "max_features": [0.3, 0.6]},
    cv=tscv, scoring="r2", n_jobs=1)
grid.fit(X_tr, y_tr)
print("Best Random Forest parameters:", grid.best_params_)
print(f"Best CV R2: {grid.best_score_:.4f}")

rf_tuned = grid.best_estimator_
pred_ret["Random Forest (tuned)"] = rf_tuned.predict(X_te)
models["Random Forest (tuned)"] = rf_tuned
results.loc["Random Forest (tuned)"] = pd.Series(
    score_row("Random Forest (tuned)", y_te.values, pred_ret["Random Forest (tuned)"], test["close"].values)).drop("Model")
results = results.sort_values("R2_return", ascending=False)
display(results)

### 7.6 What drives the predictions? Feature importance

In [ ]:
rf = models["Random Forest"]; gb = models["Gradient Boosting"]
perm = permutation_importance(rf, X_te, y_te, n_repeats=10, random_state=RANDOM_STATE, scoring="r2", n_jobs=1)

imp = pd.DataFrame({"RF impurity": pd.Series(rf.feature_importances_, FEATURES),
                    "GB impurity": pd.Series(gb.feature_importances_, FEATURES),
                    "RF permutation (test)": pd.Series(perm.importances_mean, FEATURES)}).sort_values("RF impurity", ascending=False)
lin_coef = pd.Series(models["Linear Regression"][-1].coef_, FEATURES).rename("LR std. coefficient")

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
imp["RF impurity"].sort_values().plot.barh(ax=axes[0], color=GOLD, title="Random Forest - impurity importance")
imp["RF permutation (test)"].sort_values().plot.barh(ax=axes[1], color="teal", title="Random Forest - permutation importance (test R2 drop)")
lin_coef.sort_values().plot.barh(ax=axes[2], color="steelblue", title="Linear Regression - standardised coefficients")
plt.tight_layout(); plt.show()
display(imp.round(4).head(10))

### 7.7 Actual vs predicted: price, return and residuals

In [ ]:
price_true = test["close"] * np.exp(y_te)
pp = {n: test["close"] * np.exp(pred_ret[n]) for n in ("Linear Regression", "Random Forest", "Gradient Boosting")}

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
ax = axes[0, 0]
zoom = test.index[-120:]
ax.plot(zoom, price_true.loc[zoom], "k", lw=1.5, label="Actual")
for n, c in zip(pp, ("#1f77b4", "#d62728", "#2ca02c")):
    ax.plot(zoom, pp[n].loc[zoom], c, lw=1, label=n)
ax.set(title="Next-day close, last 120 test days", ylabel="USD / oz"); ax.legend()

ax = axes[0, 1]
ax.scatter(y_te, pred_ret["Random Forest"], s=6, alpha=0.4, color="#d62728")
lim = max(abs(y_te).max(), abs(pred_ret["Random Forest"]).max())
ax.plot([-lim, lim], [-lim, lim], "k--", lw=0.8)
ax.set(title=f"Random Forest: predicted vs actual return (R2 = {results.loc['Random Forest', 'R2_return']:.3f})",
       xlabel="Actual log-return", ylabel="Predicted log-return")

ax = axes[1, 0]
resid = price_true - pp["Random Forest"]
ax.plot(resid.index, resid, color="grey", lw=0.7); ax.axhline(0, color="k")
ax.set(title="Random Forest price residuals (actual - predicted)", ylabel="USD")

ax = axes[1, 1]
roll_acc = pd.Series((np.sign(pred_ret["Random Forest"]) == np.sign(y_te.values)).astype(float), index=test.index).rolling(60).mean() * 100
ax.plot(roll_acc.index, roll_acc, color="#d62728", label="Random Forest")
ax.axhline(50, color="k", ls="--", lw=0.8)
ax.set(title="Rolling 60-day directional accuracy (%)"); ax.legend()
plt.tight_layout(); plt.show()

### 7.8 Does a tiny edge translate into a trading strategy?
A simple long/flat strategy: hold gold the next day only if the model predicts a positive return. Transaction cost = 1 basis point per position change. This is an illustration, not trading advice.

In [ ]:
COST = 0.0001
bt = pd.DataFrame({"actual": y_te.values}, index=test.index)
for n in ("Linear Regression", "Random Forest", "Gradient Boosting"):
    sig = (pd.Series(pred_ret[n], index=test.index) > 0).astype(int)
    turnover = sig.diff().abs().fillna(0)
    bt[f"{n}"] = sig * bt["actual"] - turnover * COST
bt["Buy & Hold"] = bt["actual"]
equity = np.exp(bt.drop(columns="actual").cumsum())

ax = equity.plot(figsize=(13, 4.8), lw=1.3, title="Cumulative growth of 1 unit (log-return strategies, test period)")
ax.set_ylabel("Growth multiple"); plt.show()

def sharpe(x): return x.mean() / x.std() * np.sqrt(252)
strat_tbl = pd.DataFrame({"Total return": equity.iloc[-1] - 1,
                          "Sharpe (ann.)": bt.drop(columns="actual").apply(sharpe),
                          "Max drawdown": (equity / equity.cummax() - 1).min()})
display(strat_tbl.round(3))

**Caution.** These curves come from one 5-year test window in which gold rose strongly, and a long-biased rule benefits from that drift. Differences between strategies are not statistically tested and would likely shrink with realistic costs, so they should not be read as proof of an exploitable edge.

### 7.9 Illustrative forecast for the next trading day
The selected model is refit on **all** available data and applied to the most recent bar. Treat this as a demonstration of the pipeline, not a signal.

In [ ]:
latest = latest_row.copy()
latest["close_pos"] = latest["close_pos"].fillna(0.5)
final_model = GradientBoostingRegressor(n_estimators=200, max_depth=2, learning_rate=0.03, subsample=0.7, random_state=RANDOM_STATE)
final_model.fit(feat[FEATURES], feat["target_ret"])
r_hat = float(final_model.predict(latest[FEATURES])[0])
last_close = float(latest["close"].iloc[0])
print(f"Last close ({latest.index[0].date()}): {last_close:,.1f}")
print(f"Predicted next-day log-return: {r_hat:+.4%}   ->   model price: {last_close * np.exp(r_hat):,.1f}")
print(f"Typical 1-day move (1 sd): +/- {feat['target_ret'].std():.2%} ~ +/- {last_close * feat['target_ret'].std():,.0f} USD")

---
## 8. Exporting the Analytics Layer for the Power BI Dashboard

The tables below are written to `powerbi_data/` and used by the dashboard described in **`Power_BI_Dashboard_Guide.md`**. Each table is tidy (one row per observation), uses ISO dates and has Power BI-friendly column names.

In [ ]:
# 1) Daily fact table ---------------------------------------------------------
daily = gold.reset_index().copy()
daily["Candle"] = np.where(daily["Close"] >= daily["Open"], "Up", "Down")
daily["Return_pct"] = daily["Return"] * 100
daily["Drawdown_pct"] = daily["Drawdown"] * 100
daily["Vol_21d_Ann_pct"] = daily["Vol_21d_Ann"] * 100
daily["Year"] = daily["Date"].dt.year
daily["Quarter"] = "Q" + daily["Date"].dt.quarter.astype(str)
daily["MonthNo"] = daily["Date"].dt.month
daily["Month"] = daily["Date"].dt.strftime("%b")
daily["DayOfWeek"] = daily["Date"].dt.strftime("%a")
daily["Zero_Volume"] = daily["Zero_Volume"].astype(int)
daily["Flat_Bar"] = daily["Flat_Bar"].astype(int)
daily = daily.drop(columns=["Return", "LogRet", "Drawdown", "Vol_21d_Ann"])
daily.to_csv(OUT_DIR / "gold_daily.csv", index=False, date_format="%Y-%m-%d")

# 2) Yearly summary -----------------------------------------------------------
yearly.reset_index().round(3).to_csv(OUT_DIR / "gold_yearly.csv", index=False)

# 3) Monthly returns (heat-map matrix source) ---------------------------------
mr = m_ret.rename("Return_pct").reset_index()
mr.columns = ["MonthStart", "Return_pct"]
mr["Year"] = mr["MonthStart"].dt.year; mr["MonthNo"] = mr["MonthStart"].dt.month; mr["Month"] = mr["MonthStart"].dt.strftime("%b")
mr.round(4).to_csv(OUT_DIR / "gold_monthly_returns.csv", index=False, date_format="%Y-%m-%d")

# 4) Test-set predictions + strategy curves -----------------------------------
pred_tbl = pd.DataFrame({"Date": test.index, "Close_t": test["close"].values,
                         "Actual_Next_Close": price_true.values, "Actual_Return": y_te.values})
for n in ("Linear Regression", "Random Forest", "Gradient Boosting"):
    key = n.replace(" ", "_")
    pred_tbl[f"Pred_Return_{key}"] = pred_ret[n]
    pred_tbl[f"Pred_Close_{key}"] = pp[n].values
    pred_tbl[f"Equity_{key}"] = equity[n].values
pred_tbl["Equity_Buy_Hold"] = equity["Buy & Hold"].values
pred_tbl.to_csv(OUT_DIR / "predictions_test.csv", index=False, date_format="%Y-%m-%d")

# 5) Model metrics, feature importance, risk summary ---------------------------
results.reset_index().rename(columns={"MAPE_%": "MAPE_pct", "Directional_Acc_%": "Directional_Acc_pct"}).round(5).to_csv(OUT_DIR / "model_metrics.csv", index=False)
level_results.reset_index().round(5).to_csv(OUT_DIR / "model_metrics_price_level.csv", index=False)
imp.reset_index().rename(columns={"index": "Feature", "RF impurity": "RF_Impurity", "GB impurity": "GB_Impurity", "RF permutation (test)": "RF_Permutation"}).round(5).to_csv(OUT_DIR / "feature_importance.csv", index=False)
risk_table.reset_index().rename(columns={"index": "Metric"}).to_csv(OUT_DIR / "risk_summary.csv", index=False)

for p in sorted(OUT_DIR.glob("*.csv")):
    print(f"{p.name:<32} {len(pd.read_csv(p)):>6,} rows")

---
## 9. Discussion and Conclusion

The cell below prints the headline numbers from this run, so the narrative stays consistent with the data.

In [ ]:
rf_r, lr_r, gb_r = results.loc["Random Forest"], results.loc["Linear Regression"], results.loc["Gradient Boosting"]
print(f"Data: {len(gold):,} trading days, {gold.index.min().date()} -> {gold.index.max().date()}; "
      f"price {gold['Close'].iloc[0]:,.0f} -> {gold['Close'].iloc[-1]:,.0f} USD/oz (CAGR {cagr:.1%}, max drawdown {gold['Drawdown'].min():.1%}).")
print(f"Cleaning: {n_incons} inconsistent OHLC bars repaired; {int(gold['Zero_Volume'].sum())} zero-volume days flagged.")
print(f"Stage A (price level)  R2 - Linear: {level_results.loc['Linear Regression (price)', 'R2_test']:.4f} | "
      f"Random Forest: {level_results.loc['Random Forest (price)', 'R2_test']:.4f} | Naive: {level_results.loc['Naive (tomorrow = today)', 'R2_test']:.4f}")
print(f"Stage B (return level) R2 - Linear: {lr_r['R2_return']:.4f} | Random Forest: {rf_r['R2_return']:.4f} | Gradient Boosting: {gb_r['R2_return']:.4f}")
print(f"Price R2 after converting returns to prices - Linear: {lr_r['R2_price']:.4f} | RF: {rf_r['R2_price']:.4f} | GB: {gb_r['R2_price']:.4f}")
print(f"Directional accuracy - Linear: {lr_r['Directional_Acc_%']:.1f}% | RF: {rf_r['Directional_Acc_%']:.1f}% | GB: {gb_r['Directional_Acc_%']:.1f}%")

### 9.1 Key findings

1. **Data quality.** The file has no missing values or duplicates, but 441 bars (almost all in 2003–2011) had impossible OHLC combinations (e.g. High below Close), 418 days (mostly 2000–2007) had zero volume, and 813 bars had identical O/H/L/C values — signs of a thin early series. They were repaired and flagged rather than dropped, so the time axis stays intact.
2. **Long-run behaviour.** Gold rose roughly twenty-fold in nominal terms over the sample, with a strong 2001–2011 bull run, a four-year decline to December 2015, a recovery, and a steep acceleration in 2024–2025 (calendar-year return of about +64 % in 2025). Drawdowns were sizeable (the August 2011 to December 2015 decline, about −44 %, was the deepest), which shows that gold is not risk-free even though it is a "safe haven".
3. **Return statistics.** Daily returns are **not normal**: they show fat tails (high excess kurtosis; Jarque-Bera strongly rejects normality), so Gaussian VaR would understate tail risk. Returns themselves are almost uncorrelated, but absolute returns are clearly autocorrelated — **volatility clusters** (calm and turbulent regimes).
4. **Seasonality.** A one-way ANOVA across calendar months is not significant (p ≈ 0.65), and the heat-map shows large month-to-month swings without a stable pattern, so any calendar effect is weak relative to daily noise.
5. **Prediction – the central lesson.** On price levels, Linear Regression reaches R² ≈ 0.99+, but so does the naïve "tomorrow = today" rule, so the score reflects price persistence rather than skill. A Random Forest trained on price levels fails on the test period because it cannot extrapolate beyond the training range. Reformulating the problem as **return prediction** fixes this and gives a fair comparison: all models then have R² close to zero (about 0–3 % at best, and negative under cross-validation), which is the signature of a nearly efficient market.
6. **Model comparison.** On the hold-out set Gradient Boosting has the highest return R² (about 0.03), followed by Random Forest (about 0.01) and the linear models (about 0.01); the naïve baseline is slightly negative. These gains are tiny and **not robust**: the time-series cross-validation R² is negative for every model, the tuned Random Forest does not beat the untuned one, and directional accuracy (≈ 52–55 %) does not exceed the 55 % an "always long" rule achieves. Permutation importance on the test set gives small, noisy values, led by the close's position within the day's range and the previous day's return, which hints at short-term mean-reversion but is far from conclusive.

### 9.2 Limitations
* One asset, one regime-heavy sample; the test period (2021 onward) contains an unusual price surge.  
* No macro drivers (real yields, USD index, inflation, ETF flows, central-bank purchases) were used; adding them is the most promising extension.  
* The futures series is a continuous contract, so roll effects are not modelled; volume in early years is unreliable.  
* Backtests ignore slippage, taxes, financing and contract rolls and are illustrative only.  
* Test-set R² is a single draw; CV folds disagree with it (negative R² for every model).

### 9.3 Conclusion
The project met all its objectives: the data were audited and cleaned, explored with five `mplfinance` chart types plus advanced statistical analytics, and modelled with Linear Regression, Random Forest and Gradient Boosting under a leakage-free chronological design. The main conclusion is methodological as well as financial: **a high R² on gold prices is easy to obtain and almost meaningless; the real test is whether a model explains next-day returns beyond a naïve benchmark, and here the answer is "only marginally".** Gold's risk, however, *is* analysable — volatility clusters and tails are heavy — so the Power BI dashboard focuses on trend, risk and regime monitoring rather than point forecasts.

### 9.4 Future work
Add macro and sentiment features; model volatility explicitly (GARCH, HAR-RV); try LSTM/temporal-fusion networks and walk-forward re-training; produce probabilistic (quantile) forecasts; and stress-test strategies with realistic costs.

---
## 10. References

*(Please verify formatting against your required citation style, and add the Colab notebook and the data provider you used.)*

1. Appel, G. (2005). *Technical Analysis: Power Tools for Active Investors.* FT Press. (MACD)
2. Baur, D. G., & Lucey, B. M. (2010). Is gold a hedge or a safe haven? An analysis of stocks, bonds and gold. *The Financial Review, 45*(2), 217–229.
3. Baur, D. G., & McDermott, T. K. (2010). Is gold a safe haven? International evidence. *Journal of Banking & Finance, 34*(8), 1886–1898.
4. Bergmeir, C., & Benítez, J. M. (2012). On the use of cross-validation for time series predictor evaluation. *Information Sciences, 191*, 192–213.
5. Bollinger, J. (2001). *Bollinger on Bollinger Bands.* McGraw-Hill.
6. Breiman, L. (2001). Random forests. *Machine Learning, 45*(1), 5–32.
7. Cont, R. (2001). Empirical properties of asset returns: stylized facts and statistical issues. *Quantitative Finance, 1*(2), 223–236.
8. mplfinance contributors. *mplfinance: Matplotlib finance utilities.* https://github.com/matplotlib/mplfinance
9. Fama, E. F. (1970). Efficient capital markets: A review of theory and empirical work. *The Journal of Finance, 25*(2), 383–417.
10. Friedman, J. H. (2001). Greedy function approximation: A gradient boosting machine. *Annals of Statistics, 29*(5), 1189–1232.
11. Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering, 9*(3), 90–95.
12. Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice* (3rd ed.). OTexts. https://otexts.com/fpp3/
13. McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 56–61.
14. Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830.
15. Virtanen, P., et al. (2020). SciPy 1.0: Fundamental algorithms for scientific computing in Python. *Nature Methods, 17*, 261–272.
16. Wilder, J. W. (1978). *New Concepts in Technical Trading Systems.* Trend Research. (RSI)
17. World Gold Council. *Gold Demand Trends and Gold Price data.* https://www.gold.org
18. Reference Colab notebook supplied by the author: https://colab.research.google.com/drive/1ZIwDE_81xAesQkC_C6E-RSqCnp4E35GF